# Fine-tune intfloat/multilingual-e5-base với TripletLoss

**Dataset:** `train_v3.jsonl` (specific + vague queries, in-batch negative)

**In-batch negative:** Vague query dùng negatives từ batch thay vì tường minh. Batch size lớn (32–64) đảm bảo đủ negative signal.

**Sau train:** zip model + upload lên Google Drive

## 1. Cài đặt thư viện

In [ ]:
# !pip install -q sentence-transformers datasets torch pandas numpy

## 2. Clone repo (shallow clone — không tải full LFS/data)

In [ ]:
import subprocess, os

# === THAY ĐỔI CÁC BIẾN NÀY TRƯỚC KHI CHẠY ===
GITHUB_REPO = "PhamMinhDan/llm_provider_benchmarking_ver2"
BRANCH      = "main"
TARGET_DIR  = "/content/llm_provider_benchmarking_ver2"

# Clone nếu chưa có, pull nếu đã có
if os.path.exists(TARGET_DIR):
    print(f"Repo đã tồn tại tại {TARGET_DIR}, pulling latest...")
    subprocess.run(["git", "-C", TARGET_DIR, "pull", "origin", BRANCH], check=True)
else:
    print(f"Cloning {GITHUB_REPO} (shallow, no LFS)...")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", BRANCH,
        "--filter=blob:none",
        f"https://github.com/{GITHUB_REPO}.git",
        TARGET_DIR,
    ], check=True)

print("Done. Repo ready.")
print(f"Contents: {os.listdir(TARGET_DIR)}")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

## 4. Import & Config

In [ ]:
import json, random, time, zipfile, shutil, io
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    evaluation,
)
from datasets import Dataset
from IPython.display import clear_output, display, FileLink

# === CONFIG ===
BASE_MODEL     = "intfloat/multilingual-e5-base"
MAX_SEQ_LENGTH = 512

# Dataset — từ repo đã clone
REPO_DIR   = Path(TARGET_DIR)
DATA_DIR   = REPO_DIR / "embedding_project/data"
TRAIN_JSON = DATA_DIR / "train_v3.jsonl"
VALID_JSON = DATA_DIR / "valid_v3.jsonl"
TEST_JSON  = DATA_DIR / "test_v3.jsonl"
CORPUS_CSV = DATA_DIR / "Dataset_DATN_28k.csv"

# Google Drive output
GDRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/DATN/models/e5_base_v3_finetuned")

# Training hyperparams
EPOCHS          = 2
BATCH_SIZE      = 32          # lớn để in-batch negative hiệu quả
GRAD_ACCUM      = 2           # effective batch = 32*2 = 64
LR              = 1e-5
WARMUP_RATIO    = 0.1
TRIPLET_MARGIN  = 0.5         # tăng margin vì in-batch negative khó hơn

LOCAL_OUTPUT_DIR = Path("/content/e5_base_v3_finetuned")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Base model: {BASE_MODEL}")
print(f"Train: {TRAIN_JSON}  |  Valid: {VALID_JSON}")
print(f"GDRIVE output: {GDRIVE_OUTPUT_DIR}")

## 5. Load Dataset

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(TRAIN_JSON)
valid_raw = load_jsonl(VALID_JSON)
test_raw  = load_jsonl(TEST_JSON)

print(f"Train: {len(train_raw):,}  |  Valid: {len(valid_raw):,}  |  Test: {len(test_raw):,}")

# Thống kê nhanh
for name, recs in [("Train", train_raw), ("Valid", valid_raw), ("Test", test_raw)]:
    qt = {}
    for r in recs:
        qt[r.get("query_type", "unknown")] = qt.get(r.get("query_type","unknown"), 0) + 1
    n_hard0 = sum(1 for r in recs if r.get("n_hard", 0) == 0)
    print(f"  {name}: qtype={qt}  |  n_hard=0: {n_hard0:,} ({n_hard0/len(recs)*100:.1f}%)")

## 6. Build TripletLoss Dataset

Format: `{anchor, positive, negative}`

- `anchor`     = query (prefix `query: `)
- `positive`   = product (prefix `passage: `)
- `negative`   = neg[0] — 1 hard negative cố định/record

In [ ]:
def build_triplets(records: list[dict], min_neg: bool = False) -> list[dict]:
    """min_neg=False: giữ mọi record, dù neg rỗng (vague query dùng in-batch negative)."""
    triplets = []
    skipped = 0
    for r in records:
        query   = r.get("query", "").strip()
        pos_lst = r.get("pos", [])
        neg_lst = r.get("neg", [])
        if not pos_lst:
            skipped += 1
            continue
        pos_text = pos_lst[0].strip()
        if not pos_text:
            skipped += 1
            continue
        # neg đầu tiên (hoặc rỗng → in-batch negative sẽ xử lý)
        neg_text = neg_lst[0].strip() if neg_lst else ""
        triplets.append({
            "anchor":     f"query: {query}",
            "positive":   f"passage: {pos_text}",
            "negative":   f"passage: {neg_text}",
            "query":       r.get("query", ""),
            "product_id":  r.get("product_id", ""),
            "query_type":  r.get("query_type", "specific"),
            "n_hard":      r.get("n_hard", 0),
            "n_easy":      r.get("n_easy", 0),
        })
    return triplets, skipped

train_triplets, skipped_train = build_triplets(train_raw, min_neg=False)
valid_triplets, skipped_valid = build_triplets(valid_raw, min_neg=False)
test_triplets,  skipped_test  = build_triplets(test_raw,  min_neg=False)

n_vague = sum(1 for t in train_triplets if t["query_type"] == "vague")
n_neg_empty = sum(1 for t in train_triplets if not t["negative"].replace("passage: ", ""))
print(f"Train triplets: {len(train_triplets):,}  |  vague={n_vague:,}  |  neg_empty={n_neg_empty:,}  (skipped {skipped_train:,})")
print(f"Valid triplets: {len(valid_triplets):,}")
print(f"Test triplets:  {len(test_triplets):,}")

t = train_triplets[0]
print(f"\nSample:")
print(f"  anchor:   {t['anchor'][:80]}...")
print(f"  positive: {t['positive'][:80]}...")
print(f"  negative: {t['negative'][:80]}...")

## 7. Tạo Dataset cho sentence-transformers

In [ ]:
def to_st_dataset(triplets: list[dict]) -> Dataset:
    return Dataset.from_list([
        {"anchor": t["anchor"], "positive": t["positive"], "negative": t["negative"]}
        for t in triplets
    ])

train_ds = to_st_dataset(train_triplets)
valid_ds = to_st_dataset(valid_triplets)

print(f"Train Dataset: {len(train_ds):,}  |  Valid Dataset: {len(valid_ds):,}")

## 8. Load Model

In [ ]:
print(f"Loading {BASE_MODEL}...")
model = SentenceTransformer(BASE_MODEL, device=DEVICE)
model.max_seq_length = MAX_SEQ_LENGTH
print(f"Model loaded.  max_seq_length={model.max_seq_length}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

## 9. Triplet Evaluator

Chọn best checkpoint dựa trên valid set mỗi epoch.

In [ ]:
triplet_evaluator = evaluation.TripletEvaluator(
    anchors=   [t["anchor"]   for t in valid_triplets[:2000]],
    positives=[t["positive"] for t in valid_triplets[:2000]],
    negatives=[t["negative"] for t in valid_triplets[:2000]],
    name="valid_triplet",
    show_progress_bar=True,
)
print(f"Triplet evaluator ready ({len(valid_triplets[:2000]):,} samples)")

## 10. Training Arguments

In [ ]:
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

args = SentenceTransformerTrainingArguments(
    output_dir=str(LOCAL_OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    fp16=torch.cuda.is_available(),
    bf16=False,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    logging_steps=100,
    log_on_each_node=False,
    run_name="e5_base_v3_finetuned",
    report_to=["none"],
    seed=42,
    data_seed=42,
)

print("Training args:")
for k, v in [("epochs", EPOCHS), ("batch_size", BATCH_SIZE), ("grad_accum", GRAD_ACCUM),
             ("effective_batch", BATCH_SIZE * GRAD_ACCUM), ("lr", LR),
             ("warmup_ratio", WARMUP_RATIO), ("triplet_margin", TRIPLET_MARGIN),
             ("fp16", args.fp16)]:
    print(f"  {k}: {v}")

## 11. Loss & Trainer

In [ ]:
train_loss = losses.TripletLoss(model=model, triplet_margin=TRIPLET_MARGIN)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=train_loss,
    evaluator=triplet_evaluator,
)

print("Trainer ready. Starting training...")

## 12. Train!

In [ ]:
t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0
print(f"\nTraining done in {elapsed/60:.1f} min")

## 13. Save Model + Zip → Google Drive

In [ ]:
FINAL_DIR = LOCAL_OUTPUT_DIR / "final"
model.save(str(FINAL_DIR))
print(f"Model saved locally: {FINAL_DIR}")

# Lưu metadata
meta = {
    "base_model":     BASE_MODEL,
    "epochs":         EPOCHS,
    "batch_size":     BATCH_SIZE,
    "learning_rate":  LR,
    "warmup_ratio":   WARMUP_RATIO,
    "triplet_margin": TRIPLET_MARGIN,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_samples":  len(train_triplets),
    "valid_samples":  len(valid_triplets),
    "train_time_min": round(elapsed / 60, 1),
    "loss": "TripletLoss",
    "dataset": str(TRAIN_JSON.name),
}
with open(FINAL_DIR / "finetune_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print("Metadata saved.")

In [ ]:
# === ZIP MODEL ===
import zipfile, os
from datetime import datetime

GDRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tên file zip theo timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP_NAME  = f"e5_base_v3_finetuned_{timestamp}.zip"
ZIP_PATH  = GDRIVE_OUTPUT_DIR / ZIP_NAME

print(f"Creating zip: {ZIP_PATH}")

def zipdir(path: Path, ziph: zipfile.ZipFile):
    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = Path(root) / file
            arcname = file_path.relative_to(path.parent)
            ziph.write(file_path, arcname)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zipdir(FINAL_DIR, zf)

zip_size_mb = ZIP_PATH.stat().st_size / 1024 / 1024
print(f"Zip created: {ZIP_PATH}")
print(f"Zip size: {zip_size_mb:.1f} MB")

# Download link trực tiếp trong Colab
from google.colab import files
files.download(str(ZIP_PATH))
print("\n[Colab] Download link đã được tạo. Có thể tải trực tiếp từ Colab.")
print(f"[Drive] File đã lưu tại: {ZIP_PATH}")

## 14. Evaluate trên Test Set

Recall@K, MRR@K, NDCG@K

In [ ]:
corpus_df = pd.read_csv(CORPUS_CSV, usecols=["product_id", "searchable_text"])
corpus_df = corpus_df.dropna(subset=["product_id", "searchable_text"])
corpus_df["product_id"] = corpus_df["product_id"].astype(str)
corpus_df = corpus_df.drop_duplicates(subset="product_id", keep="first")
corpus_ids   = corpus_df["product_id"].tolist()
corpus_texts = corpus_df["searchable_text"].tolist()
print(f"Corpus: {len(corpus_ids):,} products")

In [ ]:
def batch_encode(texts: list[str], batch_size: int = 256) -> np.ndarray:
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

print("Encoding corpus...")
t0 = time.time()
corpus_emb = batch_encode([f"passage: {t}" for t in corpus_texts])
print(f"Corpus encoded: {corpus_emb.shape}  ({time.time()-t0:.1f}s)")

In [ ]:
def recall_at_k(retrieved: list[str], relevant: set, k: int) -> float:
    return sum(1 for pid in retrieved[:k] if pid in relevant) / max(len(relevant), 1)

def mrr_at_k(retrieved: list[str], relevant: set, k: int) -> float:
    for i, pid in enumerate(retrieved[:k]):
        if pid in relevant:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(retrieved: list[str], relevant: set, k: int) -> float:
    dcg  = sum(1.0 / np.log2(i + 2) for i, pid in enumerate(retrieved[:k]) if pid in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(k, len(relevant))))
    return dcg / max(idcg, 1e-9)

def retrieve_topk(query_texts: list[str], k: int = 50) -> list[list[str]]:
    q_emb = model.encode(
        [f"query: {q}" for q in query_texts],
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    scores   = np.dot(q_emb, corpus_emb.T)
    topk_idx = np.argpartition(-scores, th=k, axis=1)[:, :k]
    sorted_idx = np.argsort(-np.take_along_axis(scores, topk_idx, axis=1), axis=1)
    topk_idx = np.take_along_axis(topk_idx, sorted_idx, axis=1)
    return [[corpus_ids[j] for j in row] for row in topk_idx]

def evaluate_split(triplets: list[dict], k_values: list[int]) -> dict:
    queries   = [t["query"] for t in triplets]
    labels    = [{t["product_id"]} for t in triplets]
    retrieved = retrieve_topk(queries, k=max(k_values))
    results   = {}
    for k in k_values:
        rec  = [recall_at_k(r, l, k) for r, l in zip(retrieved, labels)]
        mrr  = [mrr_at_k(r, l, k)    for r, l in zip(retrieved, labels)]
        ndcg = [ndcg_at_k(r, l, k)   for r, l in zip(retrieved, labels)]
        results[f"Recall@{k}"]  = round(np.mean(rec)  * 100, 2)
        results[f"MRR@{k}"]    = round(np.mean(mrr)  * 100, 2)
        results[f"NDCG@{k}"]   = round(np.mean(ndcg) * 100, 2)
    return results

print(f"Evaluating on Test set ({len(test_triplets):,} samples)...")
t0 = time.time()
test_metrics = evaluate_split(test_triplets, k_values=[1, 5, 10, 20, 50])
print(f"Done in {time.time()-t0:.1f}s\n")

print("=" * 45)
print("  TEST SET RETRIEVAL RESULTS")
print("=" * 45)
for metric, value in test_metrics.items():
    print(f"  {metric:20s}: {value:6.2f}%")
print("=" * 45)

In [ ]:
# Per-query-type breakdown
print("\nPer-query-type breakdown:")
for qtype in ["specific", "vague"]:
    subset = [t for t in test_triplets if t["query_type"] == qtype]
    if not subset:
        continue
    print(f"\n  [{qtype.upper()}]  n={len(subset):,}")
    for metric, value in evaluate_split(subset, k_values=[10, 20]).items():
        print(f"    {metric:20s}: {value:6.2f}%")

In [ ]:
# Lưu kết quả đánh giá
METRICS_FILE = FINAL_DIR / "test_metrics.json"
with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2, ensure_ascii=False)
print(f"Test metrics saved: {METRICS_FILE}")

# Copy metrics lên Drive
import shutil
shutil.copy(METRICS_FILE, GDRIVE_OUTPUT_DIR / "test_metrics.json")
print(f"Copied to: {GDRIVE_OUTPUT_DIR / 'test_metrics.json'}")

---

## Tổng kết sau train

**Local (Colab):** `/content/e5_base_v3_finetuned/final/`

**Google Drive:** `/content/drive/MyDrive/DATN/models/e5_base_v3_finetuned/`

Gồm:
- `e5_base_v3_finetuned_YYYYMMDD_HHMMSS.zip` — model weights
- `test_metrics.json` — kết quả đánh giá

**Giải nén model đã train:**
```bash
unzip e5_base_v3_finetuned_YYYYMMDD_HHMMSS.zip -d /path/to/models/
```

**Sử dụng model:**
```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("/path/to/models/final")

query_emb  = model.encode(["query: sữa cho bé"], normalize_embeddings=True)
corpus_emb = model.encode([f"passage: {t}" for t in corpus_texts], normalize_embeddings=True)
scores = np.dot(query_emb, corpus_emb.T)
```